---
Window size - 20 seconds

---

Paired Wilcoxon

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "BH_DB_Features_WS20.csv"  # Update to your 15s, 20s, or 30s CSV
ALPHA     = 0.05                       # Significance threshold (5%)
BASELINE  = 'NB'
INTERVENTIONS = ['BH', 'DB']           # The two conditions to test against baseline

# Features we are explicitly ignoring
DEAD_FEATURES = [
    'LF_Delta', 'LF_HF_Ratio_Delta', 'LF_Ratio', 
    'LF_HF_Ratio_Ratio', 'Raw_LF_HF_Ratio'
]

def analyze_intervention(df, baseline, intervention, features):
    """
    Runs paired Wilcoxon tests comparing the baseline (NB) against an intervention (BH or DB).
    """
    results = []
    
    # 1. Aggregate: Get the mean feature value per Subject per Category
    subject_agg = df[df['Category'].isin([baseline, intervention])].groupby(['Subject', 'Category'])[features].mean().reset_index()
    
    # 2. Iterate through every feature
    for feat in features:
        # Pivot so we have Index=Subject, Cols=[NB, Intervention]
        # We drop NA here safely because we pre-filtered to ONLY the two classes we care about
        pivot = subject_agg.pivot(index='Subject', columns='Category', values=feat).dropna()
        
        if baseline not in pivot.columns or intervention not in pivot.columns or len(pivot) < 5:
            continue
            
        base_vals = pivot[baseline].values
        interv_vals = pivot[intervention].values
        
        # Run Paired Wilcoxon Signed-Rank Test
        try:
            stat, p_val = wilcoxon(base_vals, interv_vals)
        except ValueError:
            p_val = 1.0 
            
        results.append({
            'Feature': feat,
            f'Median_{baseline}': np.median(base_vals),
            f'Median_{intervention}': np.median(interv_vals),
            'p_value': p_val
        })
        
    res_df = pd.DataFrame(results)
    if res_df.empty:
        return res_df
        
    # 3. Apply Bonferroni Correction
    # Multiply p-value by number of features tested
    res_df['p_adjusted'] = res_df['p_value'] * len(features)
    res_df['p_adjusted'] = res_df['p_adjusted'].clip(upper=1.0)
    
    # Flag significant features
    res_df['Significant'] = res_df['p_adjusted'] < ALPHA
    res_df = res_df.sort_values('p_value').reset_index(drop=True)
    
    return res_df


def main():
    if not os.path.exists(INPUT_CSV):
        print(f"[!] ERROR: {INPUT_CSV} not found.")
        return

    print(f"Loading data from {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # Exclude non-feature columns and dead features
    non_features = ['Subject', 'Timestamp_Sec', 'Category', 'y', 'Unnamed: 0']
    features = [col for col in df.columns if col not in non_features and col not in DEAD_FEATURES]
    
    print(f"Found {len(features)} valid numerical features.")
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.float_format', lambda x: f"{x:.5f}")

    all_results = {}

    for intervention in INTERVENTIONS:
        print("\n" + "=" * 90)
        print(f"STATISTICAL TEST: {BASELINE} vs {intervention} (Paired Wilcoxon, Bonferroni Corrected)")
        print("=" * 90)
        
        res_df = analyze_intervention(df, BASELINE, intervention, features)
        
        if res_df.empty:
            print(f"[!] Not enough paired data to run statistical tests for {intervention}.")
            continue
            
        print(res_df[['Feature', f'Median_{BASELINE}', f'Median_{intervention}', 'p_value', 'p_adjusted', 'Significant']])
        print("-" * 90)
        
        sig_count = res_df['Significant'].sum()
        print(f"Conclusion: {sig_count} out of {len(features)} features are statistically significant")
        print(f"predictors of a {intervention} event.")
        
        all_results[intervention] = res_df

    # Optional: Save results to CSVs
    for intervention, res_df in all_results.items():
        output_file = f"Stats_{BASELINE}_vs_{intervention}.csv"
        res_df.to_csv(output_file, index=False)
        print(f"\nSaved {intervention} statistics to: {output_file}")

if __name__ == "__main__":
    main()

Loading data from BH_DB_Features_WS20.csv...
Found 20 valid numerical features.

STATISTICAL TEST: NB vs BH (Paired Wilcoxon, Bonferroni Corrected)
                   Feature  Median_NB  Median_BH  p_value  p_adjusted  \
0                 HF_Delta    0.00028   -0.03913  0.00000     0.00000   
1              pNN50_Delta    0.09004  -17.55641  0.00000     0.00000   
2                 HF_Ratio    0.99859    0.38238  0.00000     0.00000   
3    Shannon_Entropy_Ratio    1.00431    1.80590  0.00000     0.00000   
4              RMSSD_Delta    0.00017   -0.02060  0.00000     0.00001   
5              RMSSD_Ratio    1.00008    0.59377  0.00000     0.00009   
6    Shannon_Entropy_Delta   -0.88354  -49.68457  0.00006     0.00113   
7              pNN50_Ratio    0.99291    0.33211  0.00014     0.00279   
8            Mean_RR_Ratio    1.00293    1.03854  0.00034     0.00686   
9            Mean_RR_Delta    0.00252    0.02527  0.00060     0.01201   
10             CV_RR_Delta   -0.00021   -0.01288 

MULTICOLLINEARITY CHECK

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "BH_DB_Features_WS20.csv"  # Update this to your active CSV
CORR_THRESHOLD = 0.85                  # Flag feature pairs with correlation > 85%

CLASS_MAP = {"NB": 0, "BH": 1, "DB": 2}

# Features we are explicitly ignoring
DEAD_FEATURES = [
    'LF_Delta', 'LF_HF_Ratio_Delta', 'LF_Ratio', 
    'LF_HF_Ratio_Ratio', 'Raw_LF_HF_Ratio',
    'Unnamed: 0' # Sometimes generated by pandas to_csv
]

def main():
    if not os.path.exists(INPUT_CSV):
        print(f"[!] ERROR: {INPUT_CSV} not found.")
        return

    print(f"Loading data from {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # 1. Map to 3-Class Target Variable
    df['y'] = df['Category'].map(CLASS_MAP)
    
    # 2. Define features (ignore metadata and dead columns)
    ignore_cols = ['Subject', 'Timestamp_Sec', 'Category', 'y']
    features = [col for col in df.columns if col not in ignore_cols and col not in DEAD_FEATURES]
    
    # Drop any remaining rows with NaNs so the model can train
    df_clean = df.dropna(subset=features + ['y']).copy()
    
    print(f"Clean dataset shape: {df_clean.shape}")
    print(f"Analyzing {len(features)} active features for 3-Class distinction...\n")
    
    # =========================================================================
    # 1. MULTICOLLINEARITY (CORRELATION) ANALYSIS
    # =========================================================================
    print("=" * 65)
    print(f"MULTICOLLINEARITY CHECK (Correlation > {CORR_THRESHOLD})")
    print("=" * 65)
    
    # Calculate absolute correlation matrix
    corr_matrix = df_clean[features].corr().abs()
    
    # Select upper triangle of correlation matrix to avoid duplicates
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Find index of feature columns with correlation greater than threshold
    high_corr_pairs = []
    for col in upper.columns:
        for row in upper.index:
            val = upper.loc[row, col]
            if pd.notna(val) and val > CORR_THRESHOLD:
                high_corr_pairs.append((row, col, val))
                
    # Sort by highest correlation
    high_corr_pairs.sort(key=lambda x: x[2], reverse=True)
    
    if not high_corr_pairs:
        print("  [+] No highly correlated feature pairs found. Data is clean!")
    else:
        print(f"  [!] Found {len(high_corr_pairs)} highly correlated pairs.")
        print("      Consider dropping one feature from each pair to reduce noise.\n")
        print(f"  {'Feature A':<25} | {'Feature B':<25} | {'Correlation'}")
        print("  " + "-" * 63)
        for feat_a, feat_b, corr_val in high_corr_pairs:
            print(f"  {feat_a:<25} | {feat_b:<25} | {corr_val:.4f}")

    # =========================================================================
    # 2. RANDOM FOREST FEATURE IMPORTANCE (MULTI-CLASS)
    # =========================================================================
    print("\n" + "=" * 65)
    print("RANDOM FOREST FEATURE IMPORTANCE RANKING (NB vs BH vs DB)")
    print("=" * 65)
    
    X = df_clean[features]
    y = df_clean['y']
    
    # Train a baseline Random Forest on all 3 classes
    rf = RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced', # Automatically handles the NB majority vs BH/DB minority
        n_jobs=-1
    )
    rf.fit(X, y)
    
    # Extract and sort importances
    importances = rf.feature_importances_
    feat_imp_df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    
    # Calculate cumulative importance
    feat_imp_df['Cumulative_Imp'] = feat_imp_df['Importance'].cumsum()
    
    print(f"  {'Rank':<5} | {'Feature':<25} | {'Importance':<10} | {'Cumulative'}")
    print("  " + "-" * 63)
    
    for idx, row in feat_imp_df.iterrows():
        rank = idx + 1
        print(f"  {rank:<5} | {row['Feature']:<25} | {row['Importance']:.4f}     | {row['Cumulative_Imp']:.4f}")
        
    print("\n  [Tip] Features with < 0.0100 importance can usually be safely dropped.")
    
    # Optional: Save to CSV
    feat_imp_df.to_csv("Feature_Importance_Ranking.csv", index=False)

if __name__ == "__main__":
    main()

Loading data from BH_DB_Features_WS20.csv...
Clean dataset shape: (28090, 29)
Analyzing 20 active features for 3-Class distinction...

MULTICOLLINEARITY CHECK (Correlation > 0.85)
  [!] Found 11 highly correlated pairs.
      Consider dropping one feature from each pair to reduce noise.

  Feature A                 | Feature B                 | Correlation
  ---------------------------------------------------------------
  SDNN_Ratio                | CV_RR_Ratio               | 0.9906
  Mean_RR_Delta             | Mean_RR_Ratio             | 0.9829
  SDNN_Delta                | CV_RR_Delta               | 0.9777
  EDR_Std_Amp_Ratio         | EDR_Peak_to_Peak_Ratio    | 0.9730
  EDR_Std_Amp_Delta         | EDR_Peak_to_Peak_Delta    | 0.9695
  SDNN_Delta                | RMSSD_Delta               | 0.9404
  RMSSD_Delta               | CV_RR_Delta               | 0.9183
  SDNN_Ratio                | RMSSD_Ratio               | 0.8842
  EDR_Mean_Amp_Delta        | EDR_Mean_Amp_Ratio       

VARIANCE INFLATION FACTOR (VIF) CHECK

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "BH_DB_Features_WS20.csv"  # Update this to your 15s, 20s, or 30s CSV

# Features we are explicitly ignoring
DEAD_FEATURES = [
    'LF_Delta', 'LF_HF_Ratio_Delta', 'LF_Ratio', 
    'LF_HF_Ratio_Ratio', 'Raw_LF_HF_Ratio',
    'Unnamed: 0'
]

def check_vif(csv_path):
    if not os.path.exists(csv_path):
        print(f"[!] ERROR: {csv_path} not found.")
        return

    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)

    # Automatically grab all numerical features, ignoring metadata and dead features
    ignore_cols = ['Subject', 'Timestamp_Sec', 'Category', 'y']
    features = [col for col in df.select_dtypes(include=[np.number]).columns 
                if col not in ignore_cols and col not in DEAD_FEATURES]

    # Keep only the target features and drop rows with NaNs or infinite values
    df_clean = df[features].replace([np.inf, -np.inf], np.nan).dropna()
    print(f"Calculating VIF on {len(features)} features across {len(df_clean):,} valid rows...\n")

    # CRITICAL: We must add a constant (intercept) for accurate VIF calculation.
    # Without this, VIF assumes data is centered around zero and artificially inflates scores.
    X = add_constant(df_clean)

    # Calculate VIF for each feature
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    # Remove the 'const' intercept from the final display and sort
    vif_data = vif_data[vif_data["Feature"] != "const"].sort_values(by="VIF", ascending=False).reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────────────────────
    # CONSOLE OUTPUT
    # ─────────────────────────────────────────────────────────────────────────────
    print("=" * 65)
    print("VARIANCE INFLATION FACTOR (VIF) CHECK")
    print("=" * 65)
    print(f"  {'Feature':<30} | {'VIF Score'}")
    print("  " + "-" * 45)
    
    for _, row in vif_data.iterrows():
        feat = row["Feature"]
        vif = row["VIF"]
        
        # Add visual warnings based on standard statistical thresholds
        if vif > 10:
            flag = " ❌ (Severe)"
        elif vif > 5:
            flag = " ⚠️ (Moderate)"
        else:
            flag = " ✅ (Good)"
            
        print(f"  {feat:<30} | {vif:>8.2f} {flag}")
        
    print("\n[Interpretation Guide]")
    print(" • VIF < 5  : Excellent (No significant multicollinearity).")
    print(" • VIF 5-10 : Acceptable/Moderate (Safe for Random Forests).")
    print(" • VIF > 10 : Severe Multicollinearity (Consider dropping the feature).")
    
    # Context note for this specific dataset
    print("\n[!] Note for Delta vs Ratio Features:")
    print("    Because you have both '_Delta' and '_Ratio' versions of the same physiological metric")
    print("    (e.g., Mean_RR_Delta and Mean_RR_Ratio), you will naturally see very high VIF scores.")
    print("    Random Forests handle this perfectly well, but Linear models would fail.")

if __name__ == "__main__":
    check_vif(INPUT_CSV)

Loading data from BH_DB_Features_WS20.csv...
Calculating VIF on 20 features across 28,090 valid rows...

VARIANCE INFLATION FACTOR (VIF) CHECK
  Feature                        | VIF Score
  ---------------------------------------------
  CV_RR_Ratio                    |   201.72  ❌ (Severe)
  SDNN_Ratio                     |   197.28  ❌ (Severe)
  SDNN_Delta                     |    56.73  ❌ (Severe)
  CV_RR_Delta                    |    39.01  ❌ (Severe)
  Mean_RR_Delta                  |    37.57  ❌ (Severe)
  EDR_Std_Amp_Ratio              |    35.91  ❌ (Severe)
  Mean_RR_Ratio                  |    35.22  ❌ (Severe)
  EDR_Peak_to_Peak_Ratio         |    32.43  ❌ (Severe)
  EDR_Std_Amp_Delta              |    32.40  ❌ (Severe)
  EDR_Peak_to_Peak_Delta         |    29.16  ❌ (Severe)
  RMSSD_Delta                    |    23.54  ❌ (Severe)
  RMSSD_Ratio                    |    11.64  ❌ (Severe)
  EDR_Mean_Amp_Ratio             |     4.95  ✅ (Good)
  EDR_Mean_Amp_Delta             |    

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ❷  FEATURE DEFINITIONS (Optimized for WS20 3-Class)
# ─────────────────────────────────────────────────────────────────────────────

G_LIST = [
    # --- Primary BH Detectors ---
    "pNN50_Ratio",             # RF Rank #3 (Excellent VIF: 1.04)
    "pNN50_Delta",             # RF Rank #4 (Excellent VIF: 1.80)
    "HF_Ratio",                # RF Rank #5 (Highly sig for BH)
    
    # --- Secondary HRV / Autonomic Nervous System features ---
    "RMSSD_Ratio",             # RF Rank #8 (Replaces the highly correlated SDNN/CV_RR)
    "Mean_RR_Ratio",           # RF Rank #13 (Captures overall heart rate baseline shifts)
    "Shannon_Entropy_Ratio",   # RF Rank #14 (Low VIF, mathematically orthogonal to others)

    # --- DB Isolators (Prevents DB from being classified as BH) ---
    "EDR_Std_Amp_Ratio",       # RF Rank #1 (The absolute best DB separator)
    "EDR_Mean_Amp_Delta"       # RF Rank #9 (Low VIF: 4.77, helps confirm DB isolation)
]

---
Window size - 30 seconds

---

Paired Wilcoxon

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "BH_DB_Features_WS30.csv"  # Update to your 15s, 20s, or 30s CSV
ALPHA     = 0.05                       # Significance threshold (5%)
BASELINE  = 'NB'
INTERVENTIONS = ['BH', 'DB']           # The two conditions to test against baseline

# Features we are explicitly ignoring
DEAD_FEATURES = [
    'LF_Delta', 'LF_HF_Ratio_Delta', 'LF_Ratio', 
    'LF_HF_Ratio_Ratio', 'Raw_LF_HF_Ratio'
]

def analyze_intervention(df, baseline, intervention, features):
    """
    Runs paired Wilcoxon tests comparing the baseline (NB) against an intervention (BH or DB).
    """
    results = []
    
    # 1. Aggregate: Get the mean feature value per Subject per Category
    subject_agg = df[df['Category'].isin([baseline, intervention])].groupby(['Subject', 'Category'])[features].mean().reset_index()
    
    # 2. Iterate through every feature
    for feat in features:
        # Pivot so we have Index=Subject, Cols=[NB, Intervention]
        # We drop NA here safely because we pre-filtered to ONLY the two classes we care about
        pivot = subject_agg.pivot(index='Subject', columns='Category', values=feat).dropna()
        
        if baseline not in pivot.columns or intervention not in pivot.columns or len(pivot) < 5:
            continue
            
        base_vals = pivot[baseline].values
        interv_vals = pivot[intervention].values
        
        # Run Paired Wilcoxon Signed-Rank Test
        try:
            stat, p_val = wilcoxon(base_vals, interv_vals)
        except ValueError:
            p_val = 1.0 
            
        results.append({
            'Feature': feat,
            f'Median_{baseline}': np.median(base_vals),
            f'Median_{intervention}': np.median(interv_vals),
            'p_value': p_val
        })
        
    res_df = pd.DataFrame(results)
    if res_df.empty:
        return res_df
        
    # 3. Apply Bonferroni Correction
    # Multiply p-value by number of features tested
    res_df['p_adjusted'] = res_df['p_value'] * len(features)
    res_df['p_adjusted'] = res_df['p_adjusted'].clip(upper=1.0)
    
    # Flag significant features
    res_df['Significant'] = res_df['p_adjusted'] < ALPHA
    res_df = res_df.sort_values('p_value').reset_index(drop=True)
    
    return res_df


def main():
    if not os.path.exists(INPUT_CSV):
        print(f"[!] ERROR: {INPUT_CSV} not found.")
        return

    print(f"Loading data from {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # Exclude non-feature columns and dead features
    non_features = ['Subject', 'Timestamp_Sec', 'Category', 'y', 'Unnamed: 0']
    features = [col for col in df.columns if col not in non_features and col not in DEAD_FEATURES]
    
    print(f"Found {len(features)} valid numerical features.")
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.float_format', lambda x: f"{x:.5f}")

    all_results = {}

    for intervention in INTERVENTIONS:
        print("\n" + "=" * 90)
        print(f"STATISTICAL TEST: {BASELINE} vs {intervention} (Paired Wilcoxon, Bonferroni Corrected)")
        print("=" * 90)
        
        res_df = analyze_intervention(df, BASELINE, intervention, features)
        
        if res_df.empty:
            print(f"[!] Not enough paired data to run statistical tests for {intervention}.")
            continue
            
        print(res_df[['Feature', f'Median_{BASELINE}', f'Median_{intervention}', 'p_value', 'p_adjusted', 'Significant']])
        print("-" * 90)
        
        sig_count = res_df['Significant'].sum()
        print(f"Conclusion: {sig_count} out of {len(features)} features are statistically significant")
        print(f"predictors of a {intervention} event.")
        
        all_results[intervention] = res_df

    # Optional: Save results to CSVs
    for intervention, res_df in all_results.items():
        output_file = f"Stats_{BASELINE}_vs_{intervention}.csv"
        res_df.to_csv(output_file, index=False)
        print(f"\nSaved {intervention} statistics to: {output_file}")

if __name__ == "__main__":
    main()

Loading data from BH_DB_Features_WS30.csv...
Found 20 valid numerical features.

STATISTICAL TEST: NB vs BH (Paired Wilcoxon, Bonferroni Corrected)
                   Feature  Median_NB  Median_BH  p_value  p_adjusted  \
0                 HF_Delta    0.00034   -0.04785  0.00000     0.00000   
1                 HF_Ratio    1.00377    0.34167  0.00000     0.00000   
2              pNN50_Delta    0.02823  -15.96105  0.00000     0.00000   
3              RMSSD_Delta    0.00021   -0.01546  0.00002     0.00042   
4              RMSSD_Ratio    1.00442    0.72898  0.00048     0.00959   
5            Mean_RR_Ratio    1.00401    1.03879  0.00094     0.01882   
6            Mean_RR_Delta    0.00327    0.02678  0.00160     0.03196   
7              pNN50_Ratio    0.99419    0.37504  0.00336     0.06715   
8    Shannon_Entropy_Ratio    1.00794    1.16142  0.01853     0.37054   
9   EDR_Peak_to_Peak_Ratio    1.00083    1.14381  0.04661     0.93221   
10              SDNN_Ratio    0.99402    1.07819 

MULTICOLLINEARITY CHECK

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "BH_DB_Features_WS30.csv"  # Update this to your active CSV
CORR_THRESHOLD = 0.85                  # Flag feature pairs with correlation > 85%

CLASS_MAP = {"NB": 0, "BH": 1, "DB": 2}

# Features we are explicitly ignoring
DEAD_FEATURES = [
    'LF_Delta', 'LF_HF_Ratio_Delta', 'LF_Ratio', 
    'LF_HF_Ratio_Ratio', 'Raw_LF_HF_Ratio',
    'Unnamed: 0' # Sometimes generated by pandas to_csv
]

def main():
    if not os.path.exists(INPUT_CSV):
        print(f"[!] ERROR: {INPUT_CSV} not found.")
        return

    print(f"Loading data from {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # 1. Map to 3-Class Target Variable
    df['y'] = df['Category'].map(CLASS_MAP)
    
    # 2. Define features (ignore metadata and dead columns)
    ignore_cols = ['Subject', 'Timestamp_Sec', 'Category', 'y']
    features = [col for col in df.columns if col not in ignore_cols and col not in DEAD_FEATURES]
    
    # Drop any remaining rows with NaNs so the model can train
    df_clean = df.dropna(subset=features + ['y']).copy()
    
    print(f"Clean dataset shape: {df_clean.shape}")
    print(f"Analyzing {len(features)} active features for 3-Class distinction...\n")
    
    # =========================================================================
    # 1. MULTICOLLINEARITY (CORRELATION) ANALYSIS
    # =========================================================================
    print("=" * 65)
    print(f"MULTICOLLINEARITY CHECK (Correlation > {CORR_THRESHOLD})")
    print("=" * 65)
    
    # Calculate absolute correlation matrix
    corr_matrix = df_clean[features].corr().abs()
    
    # Select upper triangle of correlation matrix to avoid duplicates
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Find index of feature columns with correlation greater than threshold
    high_corr_pairs = []
    for col in upper.columns:
        for row in upper.index:
            val = upper.loc[row, col]
            if pd.notna(val) and val > CORR_THRESHOLD:
                high_corr_pairs.append((row, col, val))
                
    # Sort by highest correlation
    high_corr_pairs.sort(key=lambda x: x[2], reverse=True)
    
    if not high_corr_pairs:
        print("  [+] No highly correlated feature pairs found. Data is clean!")
    else:
        print(f"  [!] Found {len(high_corr_pairs)} highly correlated pairs.")
        print("      Consider dropping one feature from each pair to reduce noise.\n")
        print(f"  {'Feature A':<25} | {'Feature B':<25} | {'Correlation'}")
        print("  " + "-" * 63)
        for feat_a, feat_b, corr_val in high_corr_pairs:
            print(f"  {feat_a:<25} | {feat_b:<25} | {corr_val:.4f}")

    # =========================================================================
    # 2. RANDOM FOREST FEATURE IMPORTANCE (MULTI-CLASS)
    # =========================================================================
    print("\n" + "=" * 65)
    print("RANDOM FOREST FEATURE IMPORTANCE RANKING (NB vs BH vs DB)")
    print("=" * 65)
    
    X = df_clean[features]
    y = df_clean['y']
    
    # Train a baseline Random Forest on all 3 classes
    rf = RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced', # Automatically handles the NB majority vs BH/DB minority
        n_jobs=-1
    )
    rf.fit(X, y)
    
    # Extract and sort importances
    importances = rf.feature_importances_
    feat_imp_df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    
    # Calculate cumulative importance
    feat_imp_df['Cumulative_Imp'] = feat_imp_df['Importance'].cumsum()
    
    print(f"  {'Rank':<5} | {'Feature':<25} | {'Importance':<10} | {'Cumulative'}")
    print("  " + "-" * 63)
    
    for idx, row in feat_imp_df.iterrows():
        rank = idx + 1
        print(f"  {rank:<5} | {row['Feature']:<25} | {row['Importance']:.4f}     | {row['Cumulative_Imp']:.4f}")
        
    print("\n  [Tip] Features with < 0.0100 importance can usually be safely dropped.")
    
    # Optional: Save to CSV
    feat_imp_df.to_csv("Feature_Importance_Ranking.csv", index=False)

if __name__ == "__main__":
    main()

Loading data from BH_DB_Features_WS30.csv...
Clean dataset shape: (25290, 29)
Analyzing 20 active features for 3-Class distinction...

MULTICOLLINEARITY CHECK (Correlation > 0.85)
  [!] Found 11 highly correlated pairs.
      Consider dropping one feature from each pair to reduce noise.

  Feature A                 | Feature B                 | Correlation
  ---------------------------------------------------------------
  SDNN_Ratio                | CV_RR_Ratio               | 0.9911
  Mean_RR_Delta             | Mean_RR_Ratio             | 0.9839
  SDNN_Delta                | CV_RR_Delta               | 0.9781
  EDR_Std_Amp_Ratio         | EDR_Peak_to_Peak_Ratio    | 0.9623
  EDR_Std_Amp_Delta         | EDR_Peak_to_Peak_Delta    | 0.9570
  SDNN_Delta                | RMSSD_Delta               | 0.9302
  RMSSD_Delta               | CV_RR_Delta               | 0.9118
  EDR_Mean_Amp_Delta        | EDR_Mean_Amp_Ratio        | 0.8803
  HF_Delta                  | HF_Ratio                 

VARIANCE INFLATION FACTOR (VIF) CHECK

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "BH_DB_Features_WS30.csv"  # Update this to your 15s, 20s, or 30s CSV

# Features we are explicitly ignoring
DEAD_FEATURES = [
    'LF_Delta', 'LF_HF_Ratio_Delta', 'LF_Ratio', 
    'LF_HF_Ratio_Ratio', 'Raw_LF_HF_Ratio',
    'Unnamed: 0'
]

def check_vif(csv_path):
    if not os.path.exists(csv_path):
        print(f"[!] ERROR: {csv_path} not found.")
        return

    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)

    # Automatically grab all numerical features, ignoring metadata and dead features
    ignore_cols = ['Subject', 'Timestamp_Sec', 'Category', 'y']
    features = [col for col in df.select_dtypes(include=[np.number]).columns 
                if col not in ignore_cols and col not in DEAD_FEATURES]

    # Keep only the target features and drop rows with NaNs or infinite values
    df_clean = df[features].replace([np.inf, -np.inf], np.nan).dropna()
    print(f"Calculating VIF on {len(features)} features across {len(df_clean):,} valid rows...\n")

    # CRITICAL: We must add a constant (intercept) for accurate VIF calculation.
    # Without this, VIF assumes data is centered around zero and artificially inflates scores.
    X = add_constant(df_clean)

    # Calculate VIF for each feature
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    # Remove the 'const' intercept from the final display and sort
    vif_data = vif_data[vif_data["Feature"] != "const"].sort_values(by="VIF", ascending=False).reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────────────────────
    # CONSOLE OUTPUT
    # ─────────────────────────────────────────────────────────────────────────────
    print("=" * 65)
    print("VARIANCE INFLATION FACTOR (VIF) CHECK")
    print("=" * 65)
    print(f"  {'Feature':<30} | {'VIF Score'}")
    print("  " + "-" * 45)
    
    for _, row in vif_data.iterrows():
        feat = row["Feature"]
        vif = row["VIF"]
        
        # Add visual warnings based on standard statistical thresholds
        if vif > 10:
            flag = " ❌ (Severe)"
        elif vif > 5:
            flag = " ⚠️ (Moderate)"
        else:
            flag = " ✅ (Good)"
            
        print(f"  {feat:<30} | {vif:>8.2f} {flag}")
        
    print("\n[Interpretation Guide]")
    print(" • VIF < 5  : Excellent (No significant multicollinearity).")
    print(" • VIF 5-10 : Acceptable/Moderate (Safe for Random Forests).")
    print(" • VIF > 10 : Severe Multicollinearity (Consider dropping the feature).")
    
    # Context note for this specific dataset
    print("\n[!] Note for Delta vs Ratio Features:")
    print("    Because you have both '_Delta' and '_Ratio' versions of the same physiological metric")
    print("    (e.g., Mean_RR_Delta and Mean_RR_Ratio), you will naturally see very high VIF scores.")
    print("    Random Forests handle this perfectly well, but Linear models would fail.")

if __name__ == "__main__":
    check_vif(INPUT_CSV)

Loading data from BH_DB_Features_WS30.csv...
Calculating VIF on 20 features across 25,290 valid rows...

VARIANCE INFLATION FACTOR (VIF) CHECK
  Feature                        | VIF Score
  ---------------------------------------------
  CV_RR_Ratio                    |   342.73  ❌ (Severe)
  SDNN_Ratio                     |   318.94  ❌ (Severe)
  SDNN_Delta                     |    56.37  ❌ (Severe)
  Mean_RR_Ratio                  |    41.36  ❌ (Severe)
  Mean_RR_Delta                  |    39.97  ❌ (Severe)
  CV_RR_Delta                    |    39.35  ❌ (Severe)
  EDR_Std_Amp_Ratio              |    28.06  ❌ (Severe)
  EDR_Peak_to_Peak_Ratio         |    24.84  ❌ (Severe)
  EDR_Std_Amp_Delta              |    24.70  ❌ (Severe)
  RMSSD_Delta                    |    23.26  ❌ (Severe)
  EDR_Peak_to_Peak_Delta         |    21.85  ❌ (Severe)
  RMSSD_Ratio                    |    10.66  ❌ (Severe)
  EDR_Mean_Amp_Ratio             |     5.24  ⚠️ (Moderate)
  EDR_Mean_Amp_Delta             

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ❷  FEATURE DEFINITIONS (Optimized for WS30 3-Class)
# ─────────────────────────────────────────────────────────────────────────────

G_LIST = [
    # --- Primary BH Detectors ---
    "HF_Ratio",                # RF Rank #2: Hugely powerful in WS30 (VIF: 4.2, p=0.000 for BH)
    "pNN50_Ratio",             # RF Rank #1: The ultimate global separator (VIF: 1.07)
    "pNN50_Delta",             # RF Rank #3: Strong secondary BH detector (VIF: 1.72)
    
    # --- Secondary HRV / Autonomic Nervous System features ---
    "Mean_RR_Ratio",           # RF Rank #11: Captures strict heart rate drops (p=0.0009 for BH)
    "RMSSD_Ratio",             # RF Rank #14: Replaces SDNN/CV_RR to fix severe VIF collinearity
    "Shannon_Entropy_Ratio",   # RF Rank #13: Low VIF (3.12), excellent mathematical diversity

    # --- DB Isolators (Filters out Deep Breathing false positives) ---
    "EDR_Std_Amp_Ratio",       # RF Rank #4: The absolute best respiration separator
    "EDR_Mean_Amp_Delta"       # RF Rank #6: Perfect secondary EDR feature (Low VIF: 4.86)
]